In [ ]:
import os
import pandas as pd
import re

In [ ]:
text_data_path = '/Users/jk1/Nextcloud/temp/FOCH_HSA/HSAarch-comments.sql'
patient_data_path = '/Users/jk1/Nextcloud/temp/FOCH_HSA/HSAarch-pdate.csv'

In [ ]:
patient_data = pd.read_csv(patient_data_path)
patient_data.head()

In [ ]:
with open(text_data_path, 'r', encoding='ISO-8859-1') as file:
    data = file.read()

In [ ]:
data[:10000]

In [ ]:
# count occurences of "REM INSERTING into EXPORT_TABLE"
data.count("Insert into EXPORT_TABLE")

In [ ]:
# Regular expression pattern to match the patient ID and timestamp
pattern = r'values\s*\((\d+),to_timestamp\(\'([\d-]+\s[\d:.]+)'

# Find all occurrences of the pattern
matches = re.findall(pattern, data)

# Create a DataFrame from the extracted matches
td_df = pd.DataFrame(matches, columns=['PatientID', 'Datetime'])


In [ ]:
td_df

In [ ]:
def sql_to_pd_df(file_path):
    with open(file_path, 'r', encoding='ISO-8859-1') as file:
        data = file.read()

    # Regular expression pattern to match each insert statement
    pattern = r'Insert into EXPORT_TABLE.*?values \((\d+),to_timestamp\(\'([^\']+)\',\'[^\']*\'\),\'((?:[^\']|\'\')*)\',\s*(EMPTY_CLOB\(\)|\'(.*?)\'|TO_CLOB\(q\'\[(.*?)\'\)(?:\s*\|\|\s*TO_CLOB\(q\'\[(.*?)\'\))*)\s*\)'


    # Find all matches in the file content
    matches = re.findall(pattern, data, re.DOTALL)

    # Lists to store the extracted information
    patient_ids = []
    datetimes = []
    headers = []
    notes = []

    # Loop over the matches and store the extracted data
    for match in matches:
        patient_id = match[0]
        datetime = match[1]
        header = match[2]

        # If note is EMPTY_CLOB
        if match[3] == "EMPTY_CLOB()":
            note = ""  # No notes for EMPTY_CLOB case
        else:
            # Capture the first part of the note (from TO_CLOB or regular text)
            first_note = match[4] if match[4] else ""

            # Capture all subsequent TO_CLOB parts (excluding the first one already captured)
            additional_clobs = re.findall(r'TO_CLOB\(q\'\[(.*?)\'\)', match[3], re.DOTALL)[1:]

            # Combine all parts into a single note, replacing newlines with spaces
            all_parts = [first_note] + additional_clobs
            note = " ".join(filter(None, all_parts))
            # note = note.replace('\n', ' ')

        patient_ids.append(patient_id)
        datetimes.append(datetime)
        headers.append(header)
        notes.append(note)

    # Create a DataFrame to organize the data
    data_dict = {
        "PatientID": patient_ids,
        "Datetime": datetimes,
        "Header": headers,
        "Notes": notes
    }

    df = pd.DataFrame(data_dict)
    return df

In [ ]:
df = sql_to_pd_df(text_data_path)

In [ ]:
df

In [ ]:
diff = pd.concat([td_df, df[['PatientID', 'Datetime']]]).drop_duplicates(keep=False)

In [ ]:
diff

In [ ]:
df[df.Header == 'CRI']

In [ ]:
from fpdf import FPDF
import os

# Function to generate a PDF for each patient
def create_patient_pdf(patient_id, datetime_list, header_list, notes_list, output_dir):
    # Create a PDF object
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add a page
    pdf.add_page()

    # Set title with a standard macOS font (Arial)
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(200, 10, f'Patient ID: {patient_id}', ln=True, align='C')

    # Add notes for each datetime, header, and note
    for dt, header, note in zip(datetime_list, header_list, notes_list):
        pdf.set_font('Arial', 'B', 12)
        pdf.cell(200, 10, f'Datetime: {dt}', ln=True)
        pdf.cell(200, 10, f'Header: {header}', ln=True)
        pdf.set_font('Arial', '', 10)
        pdf.multi_cell(0, 10, f'Notes: {note}')
        pdf.ln(10)  # Add a line break between notes

    # Save the PDF to a file
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)  # Create directory if not exists
    file_path = f"output_pdfs/Patient_{patient_id}.pdf"
    pdf.output(file_path)
    return file_path

In [ ]:
# Group data by patient ID
def generate_pdfs_for_all_patients(df):
    # Group data by patient ID
    patient_data = {}

    for idx, row in df.iterrows():
        patient_id = row['PatientID']
        if patient_id not in patient_data:
            patient_data[patient_id] = {"datetimes": [], "headers": [], "notes": []}
        patient_data[patient_id]["datetimes"].append(row['Datetime'])
        patient_data[patient_id]["headers"].append(row['Header'])
        patient_data[patient_id]["notes"].append(row['Notes'])

    # Generate PDFs for each patient
    pdf_files = []
    for patient_id, data in patient_data.items():
        pdf_file = create_patient_pdf(patient_id, data["datetimes"], data["headers"], data["notes"])
        pdf_files.append(pdf_file)

    return pdf_files

In [ ]:
pdf_files = generate_pdfs_for_all_patients(df.head())
